# Datapreparation

This notebook is supposed to preprocess data and give an overview of the dataset

In [21]:
# import libraries
import os
import sys
import json
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from datasets import Dataset
from setfit import SetFitModel, Trainer, TrainingArguments

# backend/ auf den Pfad legen, damit die Projekt-Parser importierbar sind
BACKEND = Path.cwd().parents[1] / "backend"
sys.path.insert(0, str(BACKEND))
from parsers import DKB
from anonymize import pseudonymize_df, suspected_person_payees
from payee import to_merchant_key, effective_key, normalize_description

DATA_DIR = BACKEND.parent / "data"

# Geheimes Salt aus .env laden (nicht committen; per .gitignore ausgeschlossen)
load_dotenv(BACKEND.parent / ".env")
SALT = os.environ["PSEUDO_SALT"]

In [34]:
# load & anonymize data
# Große synthetische Basis (2.640 Zeilen, ~80 Beispiele/Unterkat., 451 distinkte Händler).
# Erzeugt mit generate_synthetic_data.py -> genug Händler-Vielfalt für einen ehrlichen
# händler-gruppierten Test. Für einen echten Auszug hier einfach den Pfad tauschen.
REPO_ROOT = BACKEND.parent
STATEMENT = REPO_ROOT / "Machine-Learning/Model/clara/dkb_synth_large.csv"

df1 = DKB.load(str(STATEMENT))

# Labels (Kategorie/Unterkategorie) übernehmen -- Kategorien sind keine PII.
# Kopfzeile dynamisch (dieselbe Erkennung wie DKB.load) -> kein fixes skiprows.
raw = pd.read_csv(STATEMENT, sep=";", encoding="utf-8", skiprows=DKB.find_header_row(str(STATEMENT)))
# df1["category"] = raw["Verwendungszweck"].values
# df1["subcategory"] = raw["Unterkategorie"].values
# del raw  # Rohdaten nicht unnötig im Speicher lassen

# Anonymisieren: bei ECHTEN Auszügen Privatkontakte in KNOWN_PERSONS eintragen.
# suspected_person_payees(df1)          # einmal ausführen, um Kandidaten zu sichten
KNOWN_PERSONS = []                       # z. B. ["Max Mustermann", "Anna Schmidt"]
df1 = pseudonymize_df(
    df1,
    salt=SALT,
    known_persons=KNOWN_PERSONS,
    redact_columns=("account",),         # ggf. "description" ergänzen, wenn Freitext PII enthält
)

display(df1)
print(df1["category"].value_counts(dropna=False))

,transaction_id,booking_date,value_date,amount,currency,payee,description,source,transaction_type,category,account
0,09ead2388b520d8f175797a59c280c9c,2026-06-15,2026-06-15,-30.00,EUR,Lucky Luke,Beitrag zum Geschenk,DKB,Ausgang,None,[redacted]
1,f9883d9164de7e43bcbe21a2c6144d1c,2026-06-15,2026-06-15,-26.39,EUR,1und1 DSL,Rechnung 06/2026 Kd-Nr. 059784,DKB,Ausgang,None,[redacted]
2,f84414c85f4d06d83ec240103ff18ad5,2026-04-13,2026-04-13,-27.41,EUR,PayPal Europe S.a.r.l. et Cie S.C.A,"9946418398416 PP.4855.PP . CITY TAXI, Ihr Eink...",DKB,Ausgang,None,[redacted]
3,856e39b4b4995cb5f6af0416b7457974,2025-10-06,2025-10-06,-25.01,EUR,Deutsche Glasfaser,Rechnung 10/2025 Kd-Nr. 951522345,DKB,Ausgang,None,[redacted]
4,de80476ed5213c416f98cb62c30d188f,2026-03-16,2026-03-16,-22.75,EUR,Spargelhof.Klein.0047/Düsseldorf,SPARGELHOF SAGT DANKE. 2026-03-16T22:18:34 KFN...,DKB,Ausgang,None,[redacted]
...,...,...,...,...,...,...,...,...,...,...,...
2636,891dc9bd5f32eb6ea3ca85ed348655c2,2026-06-07,2026-06-07,-33.16,EUR,O2 DSL,Rechnung 06/2026 Kd-Nr. 488206,DKB,Ausgang,None,[redacted]
2637,a8a2b5ffaca152f6bd9184eda86d619d,2025-09-06,2025-09-06,-32.40,EUR,CleverShuttle/Nürnberg,Kartenzahlung 2025-09-06T07:29:21 KFN 0 VJ 2812,DKB,Ausgang,None,[redacted]
2638,7dc0755a98470b6a51e95c9920807472,2025-12-03,2025-12-03,-31.54,EUR,Alnatura/Dortmund,ALNATURA 38721489//Dortmund/DE 2025-12-03T17:2...,DKB,Ausgang,None,[redacted]
2639,1ac4a6ccc5f8be1abdf64e01e71814a0,2025-12-24,2025-12-24,-138.94,EUR,EWE Gas,Abschlag 12/2025 Kundennr 185114777,DKB,Ausgang,None,[redacted]


category
None    2641
Name: count, dtype: int64


In [35]:
# payee -> effektiver Händler-Key (von Memory UND ML-Fallback genutzt).
# Bei Zahlungsdienstleistern (PayPal) den echten Händler aus der description ziehen,
# sonst den payee bereinigen.
df1["payee_norm"] = df1.apply(lambda r: effective_key(r["payee"], r["description"]), axis=1)

# Verwendungszweck als eigenes Text-Feature (Wortgrenzen bleiben, Referenz-IDs raus).
# Originalspalte bleibt erhalten -- effective_key liest weiterhin von dort.
df1["description_norm"] = df1["description"].map(normalize_description)

with pd.option_context("display.max_colwidth", 40):
    display(df1[["payee", "payee_norm", "description", "description_norm"]].head(10))

,payee,payee_norm,description,description_norm
0,Lucky Luke,luckyluke,Beitrag zum Geschenk,beitrag zum geschenk
1,1und1 DSL,unddsl,Rechnung 06/2026 Kd-Nr. 059784,rechnung kd nr
2,PayPal Europe S.a.r.l. et Cie S.C.A,citytaxi,9946418398416 PP.4855.PP . CITY TAXI...,pp pp city taxi ihr einkauf bei city...
3,Deutsche Glasfaser,deutscheglasfaser,Rechnung 10/2025 Kd-Nr. 951522345,rechnung kd nr
4,Spargelhof.Klein.0047/Düsseldorf,spargelhofklein,SPARGELHOF SAGT DANKE. 2026-03-16T22...,spargelhof sagt danke kfn vj
5,Rossmann Babywelt,rossmannbabywelt,Lastschrift 10/2025 Ref 170598445,lastschrift ref
6,wilhelm.tel,wilhelmtel,Rechnung 06/2026 Kd-Nr. 7801104,rechnung kd nr
7,KVB Köln,kvb,Lastschrift 09/2025 Ref 79736383,lastschrift ref
8,PayPal Europe S.a.r.l. et Cie S.C.A,taxigenossenschaft,6355373096496 PP.4855.PP . TAXI GENO...,pp pp taxi genossenschaft ihr einkau...
9,PayPal Europe S.a.r.l. et Cie S.C.A,adlerapotheke,1584668793030 PP.4855.PP . ADLER APO...,pp pp adler apotheke ihr einkauf bei...


In [36]:
## Zeilen ohne Gläubiger-ID und Mandatsreferenz (z. B. PayPal) sichten, um ggf. weitere Pseudonymisierungen vorzunehmen.
raw[raw[["Gläubiger-ID", "Mandatsreferenz"]].fillna("").apply(lambda s: s.str.strip().eq("")).all(axis=1)][["Zahlungsempfänger*in", "Verwendungszweck", "Betrag (€)"]]
# alle Zeilen ohne Gläubiger-ID und Mandatsreferenz (z. B. PayPal) sichten, um ggf. weitere Pseudonymisierungen vorzunehmen.

,Zahlungsempfänger*in,Verwendungszweck,Betrag (€)
4,Spargelhof.Klein.0047/Düsseldorf,SPARGELHOF SAGT DANKE. 2026-03-16T22:18:34 KFN...,"-22,75"
10,Hotel.Astoria.124/Köln,HOTEL 2025-08-14T16:29:42 KFN 0 VJ 2812,"-734,43"
13,Book.Depository.8116/Stuttgart,Kartenzahlung 2026-02-08T07:18:37 KFN 0 VJ 2812,"-38,10"
14,FREENOW/Stuttgart,FREENOW SAGT DANKE. 2025-09-21T13:55:06 KFN 0 ...,"-38,28"
15,TEDi/Frankfurt,TEDI 66526834//Frankfurt/DE 2026-02-06T11:32:5...,"-13,59"
...,...,...,...
2626,Marktstand.Bio-Ecke.146/Freiburg,Kartenzahlung 2026-02-25T22:33:26 KFN 0 VJ 2812,"-36,95"
2628,famila.712/Dortmund,FAMILA SAGT DANKE. 2026-04-09T22:59:45 KFN 0 V...,"-7,22"
2631,üstra.Hannover.1912/Bonn,ÜSTRA SAGT DANKE. 2026-04-22T19:03:40 KFN 0 VJ...,"-38,57"
2637,CleverShuttle/Nürnberg,Kartenzahlung 2025-09-06T07:29:21 KFN 0 VJ 2812,"-32,40"


In [38]:
# zeige die Anzahl der Zeilen pro Händler (payee_norm) an, um zu sehen, welche Händler viele oder wenige Beispiele haben.
display(df1["payee_norm"].value_counts())

payee_norm
ardzdfdeutschlandradiobeitragsservice    80
stadtwerke                               12
ca                                        9
freenow                                   8
uber                                      8
                                         ..
luckyluke                                 1
peekundcloppenburg                        1
dmdrogerie                                1
zweiradst                                 1
mullerdrogerie                            1
Name: count, Length: 473, dtype: int64